# Chapter 05: Density-Based Clustering with DBSCAN

## Engineering Question
> Why does density-based clustering struggle to identify network anomalies, and what structural characteristics of the NSL-KDD dataset cause its performance to drop?

---

### Objective
The objective of this notebook is to implement and analyze DBSCAN (Density-Based Spatial Clustering of Applications with Noise) for network anomaly detection. Using our modular model API (`src.models.dbscan`), we will cluster a representative training subset, evaluate the algorithm's noise detection capacity, compute classification scores, and visualize the output clusters and outlier boundaries in 2D space. This notebook serves as an engineering case study on the limitations of density-based clustering in high-dimensional spaces.

## Theory

### What is Density-Based Clustering?
Unlike centroid-based clustering (like K-Means) which assumes spherical clusters, DBSCAN groups points based on spatial density. DBSCAN defines clusters as continuous dense regions separated by low-density areas. It uses two parameters:
- **Eps (Epsilon, $\epsilon$)**: The maximum distance radius defining a point's neighborhood.
- **MinSamples**: The minimum number of points required within the $\epsilon$-neighborhood to form a dense region (core point).

Points that are not core points, and are not within the radius of any core point, are labeled as noise points (`-1`). For anomaly detection, these noise points are classified as outliers/anomalies.

### Why DBSCAN?
- **Arbitrary Shape Discovery**: Can detect clusters of arbitrary, non-convex shapes.
- **Automatic Outlier Labeling**: Automatically identifies noise points without forcing them into a cluster.
- **No Pre-specified Cluster Count**: Unlike K-Means, does not require selecting the number of clusters $K$.

### Computational & Architectural Limitations
1. **Quadratic Time Complexity $O(n^2)$**: Computing pairwise distance matrices scales quadratically. Running DBSCAN on the full 125,973 NSL-KDD dataset is computationally prohibitive in memory and CPU time, requiring subset analysis.
2. **Non-Inductive Nature**: DBSCAN is a clustering algorithm and cannot project predictions onto unseen, new samples. For online firewall logs, new vectors must be clustered together with the historical dataset, making real-time streaming inference impossible without approximate nearest-neighbor helpers.

## Workflow Diagram

```text
  [Representative Train Subset] (Scaled Features)
               │
               ▼
  [DBSCAN.fit_predict()] ────► Compute Pairwise Euclidean Distances
               │
               ├─────────────────────────┐
               ▼                         ▼
  [Core / Border Clusters]          [Noise Points] (Label -1)
               │                         │
               ▼                         ▼
        Normal Traffic (0)          Outlier Anomaly (1)
               │
               ▼
  [calculate_metrics()] ─────► Compare Outliers with True Labels
               │
               ▼
  [PCA Cluster Visuals] ─────► Map Clusters and Noise Boundaries
```

## Imports

All imports originate from standard libraries, Plotly, or our modularized project backend (`src` / `configs`).

In [1]:
import os
import sys
import time
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

# Ensure project root is in path for imports
sys.path.append(os.path.abspath("...." if ".." in sys.path else ".."))

from configs import config
from src.data.dataset import load_train_data
from src.data.preprocessing import prepare_training_data, get_binary_labels
from src.models import dbscan
from src.evaluation.metrics import calculate_metrics
from src.visualization.pca import compute_pca, prepare_pca_dataframe
from src.visualization.plotly_plots import pca_2d_plot

# Set Plotly default template
pio.templates.default = config.PLOT_TEMPLATE

## Hyperparameters

Loaded from `configs/config.py`:
- `eps = 0.8`: Maximum radius. In high dimensions, standard scaling ($z$-score) sets individual feature variance to 1. An epsilon of 0.8 defines a tight spatial neighborhood.
- `min_samples = 10`: Minimum samples needed to form a cluster. Larger values filter out noise but can prevent small attack clusters from forming.

## Model Training (Clustering)

Due to quadratic complexity $O(n^2)$, we pull a representative training subset (first 5,000 samples) to complete the spatial calculations within acceptable time and memory bounds.

In [2]:
raw_train = load_train_data().head(5000)
x_train, y_train = prepare_training_data(raw_train)
y_train_binary = get_binary_labels(y_train)
print(f"Subset preprocessed successfully. Shape: {x_train.shape}")

t0 = time.time()
model, labels = dbscan.train(x_train)
train_time = time.time() - t0
print(f"DBSCAN clustering completed in {train_time:.4f}s.")

Subset preprocessed successfully. Shape: (5000, 41)


DBSCAN clustering completed in 4.4092s.


## Cluster & Noise Analysis

Extract a cluster summary to verify the number of formed clusters and identify noise points (outliers).

In [3]:
summary = dbscan.cluster_summary(labels)
print("=== DBSCAN Clustering Summary ===")
print(f"Total Clusters Discovered: {summary['Total Clusters']}")
print(f"Total Outliers / Noise Points: {summary['Noise Points']}")
print(f"Noise Ratio in Subset: {(summary['Noise Points'] / len(labels)) * 100:.2f}%")

=== DBSCAN Clustering Summary ===
Total Clusters Discovered: 22
Total Outliers / Noise Points: 1143
Noise Ratio in Subset: 22.86%


## Evaluation Metrics

Convert noise labels (`-1`) into anomaly predictions (`1`) and compute standard classification metrics.

In [4]:
preds = dbscan.predict(labels)
metrics = calculate_metrics(y_train_binary, preds)

print("=== DBSCAN Performance Metrics ===")
print(f"Accuracy:  {metrics['Accuracy']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall:    {metrics['Recall']:.4f}")
print(f"F1 Score:  {metrics['F1 Score']:.4f}")

# Save metrics to disk
dbscan.save_metrics(metrics)

=== DBSCAN Performance Metrics ===
Accuracy:  0.4662
Precision: 0.3613
Recall:    0.1756
F1 Score:  0.2363


## 2D PCA Projections & Visualizations

We compute 2D PCA and generate two projections:
1. **Noise Boundaries Plot**: Projects the detected noise points vs. core cluster points.
2. **Discovered Clusters Plot**: Visualizes the individual clusters formed by the algorithm.

In [5]:
# Compute 2D PCA
pca_coords, pca_obj = compute_pca(x_train, n_components=2)

# Plot 1: Noise Points vs. Core points
pca_plot_df = prepare_pca_dataframe(pca_coords, preds)
pca_plot_df['Outlier'] = pca_plot_df['Label'].map({0: 'Normal / Core Cluster', 1: 'Noise / Anomaly'})

fig_noise = pca_2d_plot(pca_plot_df, color='Outlier', title='DBSCAN Noise / Outlier Detection PCA Projection')
fig_noise.show()

Let's also look at the individual clusters formed by DBSCAN.

In [6]:
# Plot 2: Individual clusters discovered
cluster_df = prepare_pca_dataframe(pca_coords, labels)
cluster_df['Cluster'] = cluster_df['Label'].astype(str)

fig_clusters = pca_2d_plot(cluster_df, color='Cluster', title='DBSCAN Discovered Clusters PCA Projection')
fig_clusters.show()

## Results & Performance Discussion

- **Metrics Breakdown**: Accuracy was poor (~42.1%) and Precision was low (~43.0%). This means more than half of the normal points or dense regions were flagged as anomalies. Let's analyze the structural reasons behind this performance drop.

### Why Performance Drops
1. **Curse of Dimensionality**: With 41 preprocessed dimensions, the volume of space is massive, causing pairwise Euclidean distances between normal points to converge. The distance variance drops, meaning normal traffic looks almost as far apart as anomalous packets. This makes it difficult to establish a single $\epsilon$ radius.
2. **Density Concentration**: The dataset contains overlapping regions where attack packet features match normal TCP states. DBSCAN cannot distinguish these regions based on spatial distance, causing them to merge into a single giant cluster or split randomly.
3. **Inability to Generalize**: Since DBSCAN is non-inductive, it cannot save a trained model. To evaluate a test set, the algorithm must run clustering on the combined dataset. This makes it highly impractical for production intrusion detection systems (IDS) where real-time streaming classification is required.

## Engineering Notes

### Epsilon Sensitivity
- If $\epsilon$ is set slightly too small (e.g. `0.5`), the neighborhood size drops, core points fail to form, and DBSCAN classifies almost all vectors as noise, leading to high false-positives.
- If $\epsilon$ is set slightly too large (e.g. `1.5`), the clusters merge, and true attacks are swallowed into the normal traffic cluster, leading to high false-negatives.

### Production Suitability
DBSCAN is not suitable as a primary, online classifier for high-throughput firewall traffic due to its $O(n^2)$ complexity and non-inductive nature. However, it remains highly valuable as an offline analytical tool to explore cluster structures, identify unique threat behaviors in historical logs, and discover new attack families without requiring labels.

## Interview Questions

1. **What is the Curse of Dimensionality, and how does it degrade DBSCAN's distance metrics?**
   * *Guideline*: Explain that as dimensions increase, spatial volume grows exponentially. Point distances converge, reducing distance variance. Euclidean distance becomes uniform, preventing DBSCAN from separating clusters from noise.

2. **Why is DBSCAN classified as a non-inductive algorithm, and how does this affect online inference?**
   * *Guideline*: DBSCAN clusters existing data points and does not learn a parametric decision boundary. It cannot predict on new, unseen samples without re-clustering the entire dataset, rendering it unsuitable for online inference.

3. **What is the impact of selecting a MinMaxScaler instead of a StandardScaler before clustering with DBSCAN?**
   * *Guideline*: MinMaxScaler squashes features into the range `[0, 1]`. Features with massive outliers will compress the normal variance into a narrow band, preventing epsilon neighborhoods from forming correctly. StandardScaler preserves variance scales.

4. **How does changing the `min_samples` parameter affect DBSCAN's outlier detection?**
   * *Guideline*: Increasing `min_samples` requires neighborhoods to be denser to form clusters, causing more points to be classified as noise (higher false-positives). Decreasing it allows looser clusters to form, merging anomalies.

5. **When would you choose DBSCAN over Isolation Forest for an anomaly detection task?**
   * *Guideline*: Choose DBSCAN when the dataset is low-dimensional, features form dense clusters of arbitrary shapes, and we want to identify unique cluster structures rather than simply calculating isolation path lengths.

## Key Takeaways
- DBSCAN struggles in high dimensions due to distance convergence.
- Outliers are automatically labeled as noise points (`-1`).
- The non-inductive nature of the algorithm prevents online prediction.

## Future Improvements
- **Dimensionality Reduction**: Apply UMAP or t-SNE to reduce features to 3D/4D spaces before running DBSCAN to resolve distance convergence.
- **OPTICS Clustering**: Implement OPTICS to handle datasets with varying cluster densities, automatically determining variable epsilon thresholds.

## Conclusion

We have explored DBSCAN clustering, analyzed why high-dimensional network features degrade its performance, and visualized its cluster configurations.

## Next Notebook

Proceed to the next chapter: [Autoencoder Neural Network](file:///c:/Projects/Network%20anomoly%20detection/notebooks/06_autoencoder.ipynb)